# Imports

In [123]:
import pandas as pd
import spacy
import numpy as np

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [44]:
df = pd.read_json("news_dataset.json")

df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


# Preprocessing

In [45]:
df.columns.isnull()

array([False, False])

## Changing Categorical to Numerical

In [46]:
df.category.unique()

array(['SCIENCE', 'BUSINESS', 'CRIME', 'SPORTS'], dtype=object)

In [47]:
law = {"SCIENCE" : 0, "BUSINESS" : 1, "CRIME" : 2, "SPORTS" : 3}

for i,k in enumerate(df.category):
    df.category.values[i] = law.get(k)

## Loading and running through the NLP pipeline

In [49]:
nlp = spacy.load("en_core_web_sm")

In [50]:
df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,0
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,0
2,Entrepreneurs Today Don't Need a Big Budget to...,1
3,These Roads Could Recharge Your Electric Car A...,1
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,2


In [51]:
def preprocess(text):

    doc = nlp(text)

    li = []

    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        
        li.append(token.lemma_)
    
    return " ".join(li)

In [52]:
df["processed_text"] = df.text.apply(preprocess)
df["vector"] = df["text"].apply(lambda x: nlp(x).vector)

## Splitting Dataset into x, y and train, test datasets

In [113]:
x_train, x_test , y_train, y_test = train_test_split(df.vector.values, df.category.values, test_size=0.2, random_state=2023)

## OverSampling

In [114]:
x_test = np.stack(x_test)
x_train = np.stack(x_train)
y_train = np.stack(y_train)
y_test = np.stack(y_test)

In [115]:
print(sorted(Counter(y_train).items()))


[(0, 1118), (1, 3415), (2, 2279), (3, 3344)]


In [103]:
smt = RandomOverSampler(random_state=70)
x_train, y_train = smt.fit_resample(x_train, y_train)

## Scaling

In [116]:
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.fit_transform(x_test)

# Model

In [124]:
model = SVC()
model.fit(x_train,y_train)

SVC()

# Evaluation

In [125]:
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.38      0.01      0.02       263
           1       0.54      0.79      0.64       839
           2       0.78      0.34      0.48       614
           3       0.54      0.68      0.60       823

    accuracy                           0.57      2539
   macro avg       0.56      0.46      0.44      2539
weighted avg       0.58      0.57      0.52      2539

